In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
from datetime import datetime, timedelta
import re
import os

# ===================== CONFIGURATION =====================

# Weather.gov stations for three major US cities
STATIONS = {
    "New York": {
        "station_id": "KNYC",  # Central Park
        "wfo": "okx",  # Weather Forecast Office code
        "timezone": "America/New_York"
    },
    "Los Angeles": {
        "station_id": "KLAX",  # Los Angeles International Airport
        "wfo": "lox",  # Los Angeles/Oxnard
        "timezone": "America/Los_Angeles"
    },
    "Chicago": {
        "station_id": "KORD",  # O'Hare International Airport
        "wfo": "lot",  # Chicago
        "timezone": "America/Chicago"
    }
}

# Date range: Last 5 years (2020-2024)
START_YEAR = 2020
END_YEAR = 2024

# Headers to mimic a real browser
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.5",
    "Accept-Encoding": "gzip, deflate",
    "Connection": "keep-alive",
    "Upgrade-Insecure-Requests": "1"
}

# ===================== HELPER FUNCTIONS =====================

def create_session():
    """Create a requests session with proper headers and delays."""
    session = requests.Session()
    session.headers.update(HEADERS)
    return session

def fetch_monthly_data(session, city, station_id, wfo, month, year):
    """
    Fetch daily weather data for a specific month and year.
    Returns a DataFrame with the data.
    """
    # Construct the URL for the climate data
    # Weather.gov uses this format for monthly data
    url = f"https://www.weather.gov/wrh/climate?wfo={wfo}&sid={station_id}&pil=CLI&month={month}&year={year}"
    
    print(f"  Fetching {city} - {month}/{year}...")
    
    try:
        response = session.get(url, timeout=15)
        response.raise_for_status()
        
        # Parse HTML
        soup = BeautifulSoup(response.content, 'html.parser')
        
        # Find climate data table - Weather.gov has multiple tables, we need the right one
        tables = soup.find_all('table')
        
        # Look for table with climate data (usually has headers like "Day", "Max Temp", etc.)
        climate_table = None
        for table in tables:
            text = table.get_text().lower()
            if 'max temp' in text and 'min temp' in text:
                climate_table = table
                break
        
        if not climate_table:
            # Alternative: look for pre-formatted text data
            pre_tags = soup.find_all('pre')
            if pre_tags:
                return parse_preformatted_data(pre_tags, city, month, year)
            return None
        
        # Parse table
        df = pd.read_html(str(climate_table))[0]
        
        # Add metadata
        df['City'] = city
        df['Station_ID'] = station_id
        df['Month'] = month
        df['Year'] = year
        
        return df
        
    except Exception as e:
        print(f"    Error fetching {city} {month}/{year}: {e}")
        return None

def parse_preformatted_data(pre_tags, city, month, year):
    """
    Parse pre-formatted text data (fallback method).
    Weather.gov sometimes presents data in <pre> tags.
    """
    for pre in pre_tags:
        text = pre.get_text()
        lines = text.split('\n')
        
        # Look for data lines (usually contain numeric data)
        data_lines = []
        for line in lines:
            if re.match(r'^\s*\d+\s+', line):  # Starts with day number
                data_lines.append(line)
        
        if data_lines:
            # Parse the data - this is format-specific
            # Example line: "  1  64  48  0.00  0.0"
            parsed_data = []
            for line in data_lines:
                parts = re.split(r'\s+', line.strip())
                if len(parts) >= 5:
                    try:
                        day = int(parts[0])
                        max_temp = float(parts[1]) if parts[1] != 'M' else None  # M = Missing
                        min_temp = float(parts[2]) if parts[2] != 'M' else None
                        precipitation = float(parts[3]) if parts[3] != 'M' else None
                        
                        parsed_data.append({
                            'Day': day,
                            'Max_Temp_F': max_temp,
                            'Min_Temp_F': min_temp,
                            'Precipitation_in': precipitation,
                            'City': city,
                            'Month': month,
                            'Year': year
                        })
                    except ValueError:
                        continue
            
            if parsed_data:
                return pd.DataFrame(parsed_data)
    
    return None

def clean_weather_data(df):
    """Clean and standardize the weather data."""
    if df is None or df.empty:
        return df
    
    # Create a copy
    df_clean = df.copy()
    
    # Standardize column names
    df_clean.columns = [col.strip().replace(' ', '_').replace('.', '') for col in df_clean.columns]
    
    # Create Date column
    if 'Day' in df_clean.columns and 'Month' in df_clean.columns and 'Year' in df_clean.columns:
        df_clean['Date'] = pd.to_datetime(df_clean[['Year', 'Month', 'Day']].astype(str).agg('-'.join, axis=1), errors='coerce')
    
    # Rename common column patterns
    column_mapping = {
        'Max_Temp': 'Max_Temp_F',
        'Min_Temp': 'Min_Temp_F',
        'Precipitation': 'Precipitation_in',
        'Avg_Temp': 'Avg_Temp_F',
        'Departure_from_Normal': 'Temp_Departure_F',
        'HDD': 'Heating_Degree_Days',
        'CDD': 'Cooling_Degree_Days',
        'Snowfall': 'Snowfall_in',
        'Snow_Depth': 'Snow_Depth_in'
    }
    
    for old, new in column_mapping.items():
        if old in df_clean.columns:
            df_clean.rename(columns={old: new}, inplace=True)
    
    # Convert temperature columns to numeric
    temp_cols = [col for col in df_clean.columns if 'Temp' in col or 'Departure' in col]
    for col in temp_cols:
        if col in df_clean.columns:
            # Remove any non-numeric characters except decimal point and minus sign
            df_clean[col] = pd.to_numeric(df_clean[col].astype(str).str.replace(r'[^\d.-]', '', regex=True), errors='coerce')
    
    # Convert precipitation/snow to numeric
    precip_cols = [col for col in df_clean.columns if 'Precipitation' in col or 'Snow' in col]
    for col in precip_cols:
        if col in df_clean.columns:
            df_clean[col] = pd.to_numeric(df_clean[col].astype(str).str.replace(r'[^\d.-]', '', regex=True), errors='coerce')
            # Replace T (trace) with 0.001
            df_clean[col] = df_clean[col].replace({'T': 0.001, 't': 0.001})
    
    # Fill missing values
    numeric_cols = df_clean.select_dtypes(include=['float64', 'int64']).columns
    for col in numeric_cols:
        if col != 'Day':  # Don't fill day numbers
            df_clean[col] = df_clean[col].fillna(df_clean[col].median())
    
    # Remove duplicates
    if 'Date' in df_clean.columns and 'City' in df_clean.columns:
        df_clean = df_clean.drop_duplicates(subset=['Date', 'City'])
    
    # Convert to metric system (optional - for analysis)
    if 'Max_Temp_F' in df_clean.columns:
        df_clean['Max_Temp_C'] = (df_clean['Max_Temp_F'] - 32) * 5/9
    if 'Min_Temp_F' in df_clean.columns:
        df_clean['Min_Temp_C'] = (df_clean['Min_Temp_F'] - 32) * 5/9
    if 'Precipitation_in' in df_clean.columns:
        df_clean['Precipitation_mm'] = df_clean['Precipitation_in'] * 25.4
    
    return df_clean

def save_progress(data, filename="weather_data_partial.csv"):
    """Save scraped data incrementally."""
    if not data:
        return
    
    try:
        if os.path.exists(filename):
            existing = pd.read_csv(filename)
            combined = pd.concat([existing, data], ignore_index=True)
            combined.to_csv(filename, index=False)
            print(f"  Progress saved to {filename} (Total rows: {len(combined)})")
        else:
            data.to_csv(filename, index=False)
            print(f"  Initial save to {filename} (Rows: {len(data)})")
    except Exception as e:
        print(f"  Error saving progress: {e}")

# ===================== MAIN SCRAPING FUNCTION =====================

def scrape_weather_data():
    """Main function to scrape weather data from Weather.gov."""
    print("=" * 60)
    print("WEATHER.GOV DATA SCRAPER")
    print(f"Scraping data from {START_YEAR} to {END_YEAR}")
    print("=" * 60)
    
    # Create session
    session = create_session()
    
    # Store all data
    all_data = []
    
    # Loop through years and months
    for year in range(START_YEAR, END_YEAR + 1):
        for month in range(1, 13):
            print(f"\nProcessing {month:02d}/{year}")
            
            # Skip future months if we're in current year
            current_date = datetime.now()
            if year == current_date.year and month > current_date.month:
                print(f"  Skipping future month {month}/{year}")
                continue
            
            # Fetch data for each city
            monthly_data = []
            for city, info in STATIONS.items():
                df = fetch_monthly_data(
                    session, 
                    city, 
                    info['station_id'], 
                    info['wfo'], 
                    month, 
                    year
                )
                
                if df is not None and not df.empty:
                    # Clean the data
                    df_clean = clean_weather_data(df)
                    monthly_data.append(df_clean)
                    print(f"    {city}: {len(df_clean)} days collected")
                else:
                    print(f"    {city}: No data available")
                
                # Be polite - delay between requests
                time.sleep(1)
            
            # Combine monthly data
            if monthly_data:
                month_combined = pd.concat(monthly_data, ignore_index=True)
                all_data.append(month_combined)
                
                # Save progress every month
                save_progress(month_combined, "weather_data_partial.csv")
            
            # Delay between months
            time.sleep(2)
    
    # Combine all data
    if all_data:
        final_df = pd.concat(all_data, ignore_index=True)
        
        # Final cleaning
        print("\nPerforming final data cleaning...")
        final_df = clean_weather_data(final_df)
        
        # Sort by date and city
        if 'Date' in final_df.columns:
            final_df = final_df.sort_values(['Date', 'City'])
        
        # Save final dataset
        final_filename = f"weather_data_{START_YEAR}_{END_YEAR}.csv"
        final_df.to_csv(final_filename, index=False)
        
        print("\n" + "=" * 60)
        print("SCRAPING COMPLETE!")
        print("=" * 60)
        print(f"Total rows collected: {len(final_df)}")
        print(f"Date range: {final_df['Date'].min()} to {final_df['Date'].max()}")
        print(f"Cities: {final_df['City'].unique().tolist()}")
        print(f"File saved: {final_filename}")
        print("\nColumns available:")
        for col in final_df.columns:
            print(f"  - {col}")
        
        return final_df
    else:
        print("\nNo data collected!")
        return None

# ===================== DATA EXPLORATION =====================

def explore_data(df):
    """Quick exploration of the scraped data."""
    if df is None:
        print("No data to explore!")
        return
    
    print("\n" + "=" * 60)
    print("DATA EXPLORATION")
    print("=" * 60)
    
    print(f"\nDataFrame shape: {df.shape}")
    print(f"\nColumns: {df.columns.tolist()}")
    
    print("\nFirst 5 rows:")
    print(df.head())
    
    print("\nSummary statistics:")
    numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns
    print(df[numeric_cols].describe())
    
    print("\nMissing values per column:")
    print(df.isnull().sum())
    
    print("\nData by city:")
    for city in df['City'].unique():
        city_data = df[df['City'] == city]
        print(f"  {city}: {len(city_data)} rows")
    
    print("\nDate range per city:")
    if 'Date' in df.columns:
        for city in df['City'].unique():
            city_dates = df[df['City'] == city]['Date']
            print(f"  {city}: {city_dates.min()} to {city_dates.max()}")

# ===================== MAIN EXECUTION =====================

if __name__ == "__main__":
    # Start scraping
    print("Starting Weather.gov scraper...")
    print("Note: This may take 15-30 minutes depending on the amount of data.")
    print("Progress will be saved incrementally to 'weather_data_partial.csv'")
    
    # Ask for confirmation
    confirm = input("\nDo you want to start scraping? (yes/no): ").lower()
    
    if confirm in ['yes', 'y', '']:
        # Scrape data
        weather_data = scrape_weather_data()
        
        # Explore the data
        if weather_data is not None:
            explore_data(weather_data)
            
            # Save a sample for inspection
            weather_data.head(100).to_csv("weather_data_sample.csv", index=False)
            print("\nSample of 100 rows saved to 'weather_data_sample.csv'")
    else:
        print("Scraping cancelled.")

Starting Weather.gov scraper...
Note: This may take 15-30 minutes depending on the amount of data.
Progress will be saved incrementally to 'weather_data_partial.csv'



Do you want to start scraping? (yes/no):  yes


WEATHER.GOV DATA SCRAPER
Scraping data from 2020 to 2024

Processing 01/2020
  Fetching New York - 1/2020...
    New York: No data available
  Fetching Los Angeles - 1/2020...
    Los Angeles: No data available
  Fetching Chicago - 1/2020...
    Chicago: No data available

Processing 02/2020
  Fetching New York - 2/2020...
    New York: No data available
  Fetching Los Angeles - 2/2020...
    Los Angeles: No data available
  Fetching Chicago - 2/2020...
    Chicago: No data available

Processing 03/2020
  Fetching New York - 3/2020...
    New York: No data available
  Fetching Los Angeles - 3/2020...
    Los Angeles: No data available
  Fetching Chicago - 3/2020...
    Chicago: No data available

Processing 04/2020
  Fetching New York - 4/2020...
    New York: No data available
  Fetching Los Angeles - 4/2020...
    Los Angeles: No data available
  Fetching Chicago - 4/2020...
    Chicago: No data available

Processing 05/2020
  Fetching New York - 5/2020...
    New York: No data avail